In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Start Spark
spark = (
    SparkSession.builder
        .master("spark://spark-cluster-master:7077")
        .appName("JupyterLabPySparkMachineLearning")
        # limit the whole application to 12 cores
        .config("spark.cores.max", "12")
        # each executor will get 3 cores → up to 4 executors (12 / 3)
        .config("spark.executor.cores", "3")
        # memory settings stay the same
        .config("spark.executor.memory", "1g")
        .config("spark.driver.cores","2")
        .getOrCreate()
)
    
print("Spark Version: ", spark.version)

In [ ]:
# Example data
data = [
    (5.1, 3.5, 1.4, 0.2, 0.0),
    (7.0, 3.2, 4.7, 1.4, 1.0),
    (6.3, 3.3, 6.0, 2.5, 2.0),
    (5.9, 3.0, 5.1, 1.8, 2.0),
    (5.0, 3.6, 1.4, 0.2, 0.0),
    (6.7, 3.0, 5.0, 1.7, 1.0),
]
columns = ["sepal_length", "sepal_width", "petal_length", "petal_width", "label"]

df = spark.createDataFrame(data, columns)

# Characteristic vector
assembler = VectorAssembler(
    inputCols=["sepal_length", "sepal_width", "petal_length", "petal_width"],
    outputCol="features"
)
assembled = assembler.transform(df)

In [ ]:
# simple split training and test
train, test = assembled.randomSplit([0.8, 0.3], seed=42)

In [ ]:
# Fit logistic regression model
lr = LogisticRegression(maxIter=10, regParam=0.01)
model = lr.fit(train)

In [ ]:
# Predict on training data
predictions = model.transform(assembled)

In [ ]:
# Evaluate test accuracy
predictions = model.transform(test)
evaluator = MulticlassClassificationEvaluator(metricName="accuracy")
accuracy = evaluator.evaluate(predictions)

print(f"Test accuracy: {accuracy:.3f}")

In [ ]:
spark.stop()